# Streamlit Simulator Dashboard Code Explanation
### Honeywell Time-Series Temperature Alarm Predictor

This notebook provides a detailed explanation of the code inside **`app.py`**, which runs the real-time simulation dashboard.

Running a time-series model in a "live" simulator requires special considerations to avoid **data leakage** (accidentally showing the model future data) and to create a smooth, responsive user interface.

#### Core Dashboard Mechanisms Explained in this Notebook:
1. **Streamlit Session State (`st.session_state`)**: How we maintain animation playback state (Play, Pause, Reset, Current Simulation Index).
2. **On-the-Fly Feature Engineering**: Calculating lags and rolling averages using only the replayed historical window (no future leakage).
3. **Dynamic Plotly Forecast Plotting**: Plotting the rolling past trend and extending future forecast points at $t+5$, $t+15$, $t+30$, and $t+60$ minutes.
4. **Actionable Alert Engine**: Logging active alarms and displaying custom operator recommendations.

--- 
## 1. Importing Libraries & Premium Custom CSS
We use custom CSS in Streamlit to override default styles, creating dark-themed, glassmorphic metric cards that resemble modern DCS (Distributed Control System) operator screens.

In [ ]:
# Code block showing imports and page styling configuration
"""
st.set_page_config(
    page_title="Honeywell Temperature Alarm Predictor",
    page_icon="🔥",
    layout="wide"
)

# Premium custom CSS injection
st.markdown('''
<style>
    .metric-card {
        background-color: #1f2937;
        border-radius: 8px;
        padding: 15px;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        text-align: center;
        border-left: 5px solid #3b82f6;
    }
    .metric-value { font-size: 28px; font-weight: bold; color: #f3f4f6; }
    .metric-label { font-size: 14px; color: #9ca3af; }
</style>
''', unsafe_allow_html=True)
"""

--- 
## 2. On-the-Fly Feature Calculation (Preventing Data Leakage)
When a model runs live, it only has access to the *past* and the *current* tick. We cannot use pandas shifts across the entire dataset because that would leak the future.

Instead, the function `compute_features_at_time` takes only the historical window up to the current simulation tick, grabs the last 65 rows (representing the last 65 minutes of operations), and calculates the feature vector on that small slice. This matches exactly how the model will run in a real-time plant OPC server.

In [ ]:
def compute_features_at_time(df_history, feature_cols):
    # Slice the last 65 rows of history
    df_slice = df_history.tail(65).copy()
    
    # 1. Time Features
    current_time = df_slice.index[-1]
    df_slice['hour'] = current_time.hour
    df_slice['month'] = current_time.month
    df_slice['dayofweek'] = current_time.dayofweek
    
    key_cols = ['03TIC_1023.PV', '03TI_1024.PV', '03TI_1015.PV', '03PIC_1023.PV', '03TI_1081.PV']
    
    # 2. Lag Features (1 to 60 minutes)
    for col in key_cols:
        for lag in [1, 2, 5, 10, 15, 30, 60]:
            df_slice[f'{col}_lag_{lag}'] = df_slice[col].shift(lag)
            
    # 3. Rolling Features (Mean, Std, Min, Max over 10m, 30m, 60m)
    for col in key_cols:
        for window in [10, 30, 60]:
            df_slice[f'{col}_roll_mean_{window}'] = df_slice[col].rolling(window=window).mean()
            df_slice[f'{col}_roll_std_{window}'] = df_slice[col].rolling(window=window).std()
            df_slice[f'{col}_roll_max_{window}'] = df_slice[col].rolling(window=window).max()
            df_slice[f'{col}_roll_min_{window}'] = df_slice[col].rolling(window=window).min()

    # 4. Difference Features
    for col in key_cols:
        for diff in [5, 15, 30]:
            df_slice[f'{col}_diff_{diff}'] = df_slice[col] - df_slice[col].shift(diff)
            
    # Extract features for the current minute and reindex to align feature columns
    feature_row = df_slice.tail(1)
    feature_df = feature_row[feature_cols].copy()
    return feature_df

--- 
## 3. Session State & The Animation Loop
Streamlit reruns the entire script from top to bottom on any user action. To run a smooth simulation, we use `st.session_state` to store properties that survive script reruns:
- `st.session_state.sim_index`: The index of the current simulation minute.
- `st.session_state.is_playing`: True if the simulation is currently running.
- `st.session_state.alert_log`: List of triggered warnings.

When `is_playing` is active, the app sleeps for a fraction of a second (determined by the speed slider), increments `sim_index` by 1, and triggers `st.rerun()`. This creates a live-looping animation.

In [ ]:
# Code block showing the Streamlit rerun loop
"""
if st.session_state.is_playing:
    # 60x speed means 1 hour of plant data replays in 1 minute (1 tick per second)
    sleep_time = max(0.01, 1.0 / speed)
    time.sleep(sleep_time)
    st.session_state.sim_index += 1
    st.rerun()
"""

--- 
## 4. Plotly Scrolling Chart & Projections
We draw a Plotly chart showing the past 60 minutes of actual data. Relative to the current simulated minute, we draw a dotted line showing the model's forecasts at $t+5$, $t+15$, $t+30$, and $t+60$ minutes.

If any forecast point crosses the red dotted `21.0` line, the forecast marker changes color from green to yellow or red, and an alert is logged to the console.

--- 
## 5. Demo Operator Recommendations
To make the dashboard look like a real control room tool, we display specific, actionable recommendations in the operator advisory log:

1. **Emergency Critical Alert (5-minute Horizon)**: Emergency alarm predicted in 5 minutes. Actions focus on immediate manual cooling activation, checking reactor trip signals, and warning site personnel.
2. **Strategic Advisory (60-minute Horizon)**: Warns operators that a temperature rise is predicted in 1 hour. Actions focus on strategic changes: checking inlet feed flow rate (`02FI_1000.PV`) and preparing to increase refrigeration compressor suction pressure (`03PIC_1013.PV`).
3. **Tactical Warning (30-minute Horizon)**: Warns operators that a threshold crossing is predicted in 30 minutes. Actions focus on tactical loops: checking level controller stability (`03LIC_1016.PV`) and checking cooling pump status.
4. **Critical Alert (15-minute Horizon)**: Critical alarm predicted in 15 minutes. Actions focus on emergency safety steps: immediately increasing reflux flow, checking bypass valves, and alerting supervisors.